<a href="https://colab.research.google.com/github/kaushalkalas-awesome/Pothole-Detection-RF-DETR/blob/main/Pothole_Detection_RF_DETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rfdetr>=1.4.0 supervision roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="kAnFM4vuNJWm5DKRv8co")
project = rf.workspace("hacker-gs2qc").project("pothole-segmentation-g6hbh-4y7lj")
version = project.version(2)
dataset = version.download("coco-segmentation")


In [ ]:
import json

with open("/content/Pothole-Segmentation-2/train/_annotations.coco.json") as f:
    data = json.load(f)

print(f"Classes: {[cat['name'] for cat in data['categories']]}")
print(f"Train images: {len(data['images'])}")
print(f"Train annotations: {len(data['annotations'])}")

In [ ]:
from rfdetr import RFDETRSegNano

model = RFDETRSegNano()

model.train(dataset_dir="/content/Pothole-Segmentation-2", epochs=5, batch_size=4, grad_accum_steps=4)

In [ ]:
model = RFDETRSegNano(pretrain_weights="/content/output/checkpoint_best_total.pth")
model.optimize_for_inference()

In [ ]:
import supervision as sv

ds_test = sv.DetectionDataset.from_coco(
    images_directory_path="/content/Pothole-Segmentation-2/test",
    annotations_path="/content/Pothole-Segmentation-2/test/_annotations.coco.json",
    force_masks=True
)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import supervision as sv

# Define annotators
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
mask_annotator = sv.MaskAnnotator()

N = 16
L = len(ds_test)

classes = ds_test.classes

annotated_images = []

for i in random.sample(range(L), N):
    path, _, annotations = ds_test[i]
    image = Image.open(path)

    # ✅ Fixed: .infer() → .predict()
    # predict() returns sv.Detections directly, no need for from_inference()
    detections = model.predict(image, threshold=0.5)

    # Convert PIL to numpy for supervision
    image_np = np.array(image)

    # Build labels
    labels = [
        f"{classes[class_id]} {conf:.2f}"
        for class_id, conf in zip(detections.class_id, detections.confidence)
    ]

    annotated_image = image_np.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = label_annotator.annotate(annotated_image, detections, labels=labels)

    annotated_images.append(annotated_image)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, img in zip(axes.flat, annotated_images):
    ax.imshow(img)
    ax.axis("off")

plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.show()